# 04 — Dynamometer Card (Load vs Position)

**Goals**
- Q7: plot load vs position; discuss physical significance
- Q8: effect of gas/liquid ratio and mechanical failures

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from well_analysis.data import load_test_data
from well_analysis.signal import check_even_sampling
from well_analysis.signal.integration import integrate_acceleration, estimate_gravity_offset
from well_analysis.detection import detect_well_state
from well_analysis.analysis.dynamometer import (
    estimate_pump_frequency, extract_dynamometer_cards, estimate_stroke_amplitude
)
from well_analysis.viz import plot_dynamometer_card

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

df = load_test_data()
_, fs = check_even_sampling(df['Timestamp'])
accel = df['Acceleration'].values
load  = df['Load'].values

is_running = detect_well_state(accel, fs=fs)
g_offset   = estimate_gravity_offset(accel, is_running)
_, position = integrate_acceleration(accel, fs=fs, gravity_offset=g_offset)

## 1. Estimate pump frequency

In [ ]:
# Use only running samples for clean spectral estimate
pump_freq = estimate_pump_frequency(accel[is_running], fs=fs)
print(f"Pump frequency: {pump_freq:.4f} Hz  (period = {1/pump_freq:.1f} s)")

## 2. Extract individual cards and plot a representative one (Q7)

In [ ]:
# Use a short running window (e.g. ~5 minutes) for the showcase card
# TODO: update window after exploring the dataset
t_start = pd.Timestamp('2020-12-25T02:01:00', tz='UTC')
t_end   = t_start + pd.Timedelta(minutes=5)
mask = (df['Timestamp'] >= t_start) & (df['Timestamp'] < t_end) & is_running

pos_w  = position[mask]
load_w = load[mask]

cards = extract_dynamometer_cards(pos_w, load_w, fs=fs, pump_freq=pump_freq)
print(f"Extracted {len(cards)} cycles in window")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
# Overlay all cards in window, colour by cycle index
colours = cm.viridis(np.linspace(0, 1, len(cards)))
for i, card in enumerate(cards):
    ax.plot(card['pos'], card['load'], lw=0.8, color=colours[i], alpha=0.7)
ax.set_xlabel('Position (m)')
ax.set_ylabel('Load (N)')
ax.set_title('Dynamometer cards — 5 min window')
plt.tight_layout()
plt.show()

## 3. Physical significance (Q7)

The dynamometer card (load vs position) is the standard diagnostic for rod pump health:

- **Top edge (upstroke)**: polished rod carries fluid weight + rod buoyancy − friction → high load plateau.
- **Bottom edge (downstroke)**: fluid weight transferred to tubing via standing valve → low load.
- **Right rise** (bottom-stroke to top): standing valve opens, travelling valve closes.
- **Left drop** (top to bottom): travelling valve opens, standing valve closes.
- **Card area** ∝ net work done per stroke ∝ fluid production.

Deviations from the ideal parallelogram indicate:
- **Rounded corners / reduced area** → gas interference / incomplete fillage
- **Inverted or collapsed card** → rod string buckling or valve failure
- **Asymmetry** → paraffin build-up, mechanical friction

## 4. Effect of gas/liquid ratio and mechanical failures (Q8)

*(Discuss after visualising multiple time windows below)*

| Condition | Card shape change |
|-----------|------------------|
| High gas / low fillage | Rounded bottom right corner; reduced upper plateau; card area shrinks |
| Fluid pound (pump hits liquid surface) | Sudden spike at bottom of upstroke |
| Worn travelling valve | Upper load line drops during upstroke |
| Worn standing valve | Lower load line rises during downstroke |
| Paraffin / tubing hole | Altered load baseline |

In [ ]:
# TODO: plot cards from multiple time windows to compare regimes